# Best On-Device SLM for GridRoute + MazeBench -- Kaggle Training Notebook

Runs the full pipeline from `neuro-symbolic-pathfinding` end to end on a free Kaggle GPU: Phase 1 baselines (Gemma 4 E2B/E4B + AlphaMaze on GridRoute 5x5 and MazeBench), then Phase 2 training recipes on Gemma 4 E2B. See `idea.md`, `refined-idea.md`, and `experiment-plan.md` in the repo for the full plan this executes. No specific technique is the point here -- get Gemma 4 as good as possible at both benchmarks, report whichever recipe actually works.

**Built for unattended execution.** Every stage below runs through a `run_stage()` wrapper: it never raises, always logs success/failure/skip to `./results/stage_log.json`, and a later stage that depends on an earlier one's output (e.g. GRPO needing SFT's checkpoint) is skipped with a clear reason rather than crashing if that input is missing. Use **Save Version -> Save & Run All (Commit)** to run the whole thing in the background on Kaggle's servers -- close the tab, come back later, and check the final "Run Summary" cell plus `results.zip` rather than watching every cell live.

**Before running:**

1. Settings (right panel) -> Accelerator -> **GPU T4 x2** (or P100).
2. Settings -> **Internet: On** (needed for pip installs, HuggingFace downloads, and cloning the repo).
3. Give this notebook access to the repo (it's private). Two options:
   - **Recommended**: Add-ons -> Secrets -> add a secret named `GITHUB_TOKEN` with a GitHub [fine-grained personal access token](https://github.com/settings/tokens) scoped read-only to this one repo, **then toggle it ON for this specific notebook** (adding a secret to your account doesn't attach it automatically). Never paste the token directly into a cell or into chat -- Kaggle Secrets keeps it out of the notebook file entirely, and this notebook only ever reads it through `kaggle_secrets`. If a token is ever exposed anywhere else (a cell, an error message, a chat), revoke it on GitHub and issue a new one -- don't reuse it.
   - **Alternative**: zip the repo yourself, upload it as a private Kaggle Dataset, attach it to this notebook (Add data -> your dataset), and change `REPO_DIR` in the next cell to wherever Kaggle mounts it (typically `/kaggle/input/<dataset-name>`), then skip the clone cell.
4. Optional: add a second secret named `HF_TOKEN` (a HuggingFace [access token](https://huggingface.co/settings/tokens), read-only is enough), also toggled ON for this notebook, for higher Hub rate limits/faster downloads. Not required for any model used here, but the notebook will use it if present.
5. Kaggle's free tier: about 30 GPU-hours/week, 9-12h max per session. Gemma 4 **E4B** may not fit the T4 at all for *training* (its LoRA footprint has been reported elsewhere as ~17GB, over the T4's 16GB) -- the feasibility check cell below confirms this on the real hardware; E4B *baseline* eval (no LoRA) is much lighter and runs regardless. The **consistency** GRPO condition costs roughly double per step -- each full-training cell below sizes its own step count from a preceding timing-test cell's measured cost against a fixed time budget, specifically so this doesn't need a human to read a timing printout and hand-adjust the next cell.

This notebook is a thin runner around the actual project code (`eval.py`, `train_sft.py`, `train_grpo.py`) -- it doesn't reimplement any of the logic, so if you change the training/eval logic, edit those files and re-clone/re-sync rather than editing this notebook's cells directly.

In [ ]:
import os, subprocess, sys

REPO_URL = "github.com/Vedang-P/neuro-symbolic-pathfinding.git"
REPO_DIR = "/kaggle/working/neuro-symbolic-pathfinding"

# GIT_TERMINAL_PROMPT=0: if auth still fails for some other reason (token
# lacks access to this repo, expired, etc.), git fails immediately with a
# clear error instead of hanging on an interactive password prompt that has
# nowhere to go in a notebook environment.
env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True, env=env)
elif not os.path.isdir(REPO_DIR):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        # Token must go in the PASSWORD slot (after the colon), not as a bare
        # username before @ -- a bare-username URL leaves git still needing a
        # password, which it then can't prompt for here ("could not read
        # Password ... No such device or address"). "x-access-token" as the
        # username is GitHub's documented convention for token-based HTTPS auth.
        clone_url = f"https://x-access-token:{token}@{REPO_URL}"
        subprocess.run(["git", "clone", clone_url, REPO_DIR], check=True, env=env)
        print("Cloned via GITHUB_TOKEN secret.")
    except Exception as e:
        print(f"Could not clone via secret ({type(e).__name__}: {e}).")
        print("Check: the secret is attached to this notebook (Add-ons -> Secrets -> make sure")
        print("it's toggled ON for this notebook specifically), the token hasn't expired, and it")
        print("has read access to this exact repo (fine-grained tokens need per-repo access grants).")
        print("If you attached the repo as a Kaggle Dataset instead, set REPO_DIR above to its mount")
        print("path (typically /kaggle/input/<dataset-name>) and re-run this cell.")
        raise

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# Pulls in alphamaze_reference/ (github.com/menloresearch/visual-thinker) -- eval.py uses their
# real MazeBench scoring code directly from this submodule when present, falling back to a less
# faithful exact-match approach (with a loud warning) if it isn't.
subprocess.run(["git", "submodule", "update", "--init"], check=True)


In [ ]:
# Optional: read from Kaggle Secrets, never pasted here -- higher HF Hub rate
# limits/download speed, and required if any candidate model is gated.
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # some older library versions check this name
    print("HF_TOKEN set from Kaggle secret.")
except Exception as e:
    print(f"No HF_TOKEN secret found ({type(e).__name__}) -- proceeding unauthenticated "
          "(fine for these public models, just slower/rate-limited).")


In [ ]:
%pip install -q -r requirements.txt
# -U (upgrade) both unsloth AND unsloth_zoo explicitly, not just unsloth --
# older unsloth_zoo versions unconditionally import ConstantLengthDataset
# from trl.trainer.utils, which TRL 0.20.0 removed; newer unsloth_zoo
# already wraps that import in a try/except, but pip won't necessarily pull
# that fix in just from `pip install unsloth` if a same-enough version is
# already satisfied by another dependency's constraint.
%pip install -q -U unsloth unsloth_zoo


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## Run infrastructure: `run_stage()`

Every pipeline stage below goes through this. It never raises -- a stage that OOMs, hits a bug, or has a transient HF Hub hiccup is logged and skipped over, not fatal to the rest of an unattended run. `required_paths` lets a stage declare "don't even try me if this input is missing" (e.g. a GRPO stage needing the preceding SFT stage's checkpoint), so a failure shows up as a clear, attributable skip reason in the log instead of a cryptic error three layers deep in someone else's library. `steps_for_budget()` reads a prior GRPO timing-test's measured per-step cost and sizes the following full run to fit a wall-clock budget automatically, which is what makes "no human between cells" actually work -- without it, the standard advice ("check the timing printout, adjust --max_steps") requires exactly the babysitting this notebook is built to avoid.

In [ ]:
import json
import subprocess
import sys
import time
from pathlib import Path

RESULTS_DIR = Path("./results")
STAGE_LOG_PATH = RESULTS_DIR / "stage_log.json"
stage_results = json.loads(STAGE_LOG_PATH.read_text()) if STAGE_LOG_PATH.exists() else []

# Safety-ceiling timeouts (a hard kill-switch if something hangs) -- these are
# NOT the target duration for a stage, just an upper bound. Full GRPO runs
# size their own --max_steps well below their timeout via steps_for_budget().
TIMEOUT_QUICK = 30 * 60        # feasibility check, GRPO timing tests
TIMEOUT_EVAL = 60 * 60         # baseline/checkpoint evals (100 MazeBench + 50 GridRoute generations)
TIMEOUT_SFT = 90 * 60          # SFT warm-start
TIMEOUT_GRPO_FULL = 2 * 60 * 60  # full GRPO training runs, ceiling above their own time budget

# Target wall-clock budget per full GRPO training run -- lower this if your
# weekly quota is tight, raise it if you have room. Applied per condition, so
# the pricier "consistency" condition naturally gets fewer steps than
# "single"/"mixed" for the same budget, without any special-casing.
GRPO_BUDGET_MINUTES = 60


def _save_stage_log():
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    STAGE_LOG_PATH.write_text(json.dumps(stage_results, indent=2))


def run_stage(name, args, required_paths=None, timeout=None):
    """Run `python <args>` as a subprocess. Always returns, never raises."""
    print(f"\n{'='*70}\n\u25b6 {name}\n{'='*70}")
    for p in (required_paths or []):
        if not Path(p).exists():
            print(f"\u23ed  SKIPPED -- required input not found: {p}")
            stage_results.append({"stage": name, "status": "skipped", "reason": f"missing {p}"})
            _save_stage_log()
            return stage_results[-1]

    t0 = time.time()
    try:
        proc = subprocess.run([sys.executable] + args, timeout=timeout)
        elapsed_min = round((time.time() - t0) / 60, 1)
        if proc.returncode == 0:
            print(f"\n\u2705 DONE ({elapsed_min} min)")
            result = {"stage": name, "status": "ok", "elapsed_min": elapsed_min}
        else:
            print(f"\n\u274c FAILED (exit code {proc.returncode}, {elapsed_min} min) -- see output above")
            result = {"stage": name, "status": "failed", "exit_code": proc.returncode, "elapsed_min": elapsed_min}
    except subprocess.TimeoutExpired:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u23f1  TIMED OUT after {elapsed_min} min (limit {timeout/60:.0f} min)")
        result = {"stage": name, "status": "timeout", "elapsed_min": elapsed_min}
    except Exception as e:
        elapsed_min = round((time.time() - t0) / 60, 1)
        print(f"\n\u274c FAILED (exception: {e})")
        result = {"stage": name, "status": "error", "error": str(e), "elapsed_min": elapsed_min}

    stage_results.append(result)
    _save_stage_log()
    return result


def steps_for_budget(timing_dir, budget_minutes=GRPO_BUDGET_MINUTES, min_steps=20, default_steps=200):
    """Read a prior train_grpo.py run's timing.json and compute a step count
    that fits budget_minutes at that measured per-step cost. Falls back to
    default_steps if timing.json is missing (e.g. the timing test itself
    failed) -- better to attempt a bounded full run than skip it outright."""
    timing_path = Path(timing_dir) / "timing.json"
    if not timing_path.exists():
        print(f"  (no timing.json at {timing_dir}, using default {default_steps} steps)")
        return default_steps
    t = json.loads(timing_path.read_text())
    per_step = t.get("per_step_s", 0)
    if per_step <= 0:
        return default_steps
    steps = max(min_steps, int(budget_minutes * 60 / per_step))
    print(f"  (timing.json: {per_step:.1f}s/step -> {steps} steps fits a {budget_minutes}min budget)")
    return steps


## Feasibility check: does Gemma 4 E2B/E4B LoRA actually fit this GPU?

Don't assume the ~8-10GB (E2B) / ~17GB (E4B) numbers from elsewhere -- confirm on this exact hardware.

In [ ]:
run_stage("Feasibility check (all candidate models)", ["check_finetune_feasibility.py"], timeout=TIMEOUT_QUICK)


## Phase 1: baselines -- Gemma 4 E2B/E4B + AlphaMaze on MazeBench and GridRoute 5x5

AlphaMaze should land near its published 93% on MazeBench (using their real scoring code via the submodule) -- this is the harness sanity check. Gemma 4's numbers on both benchmarks are the actual open question: no SLM has been tested on GridRoute anywhere in the literature found so far.

In [ ]:
ALPHAMAZE_LOCAL_PATH = "data/models/alphamaze-v0.2-1.5b"
os.makedirs("data/models", exist_ok=True)
try:
    if not os.path.isdir(ALPHAMAZE_LOCAL_PATH):
        from huggingface_hub import snapshot_download
        snapshot_download("homebrewltd/AlphaMaze-v0.2-1.5B", local_dir=ALPHAMAZE_LOCAL_PATH)
    print("AlphaMaze checkpoint ready.")
except Exception as e:
    print(f"\u26a0\ufe0f  AlphaMaze checkpoint download failed ({e}) -- AlphaMaze-specific "
          "stages below will be skipped (they require-path-check this directory).")


In [ ]:
run_stage("AlphaMaze baseline: MazeBench",
          ["eval.py", "--model", "alphamaze", "--benchmark", "mazebench", "--n", "100",
           "--output_dir", "./results/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=TIMEOUT_EVAL)
run_stage("AlphaMaze baseline: GridRoute NL",
          ["eval.py", "--model", "alphamaze", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[ALPHAMAZE_LOCAL_PATH], timeout=TIMEOUT_EVAL)


In [ ]:
run_stage("Gemma 4 E2B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "mazebench", "--n", "100",
           "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E2B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)


E4B baseline (inference-only, no LoRA) is much lighter than E4B *training* -- run this regardless of what the feasibility check said about training, since it doesn't need Unsloth LoRA attachment, just enough VRAM to hold the weights at 4-bit.

In [ ]:
run_stage("Gemma 4 E4B baseline: MazeBench",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "mazebench", "--n", "100",
           "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)
run_stage("Gemma 4 E4B baseline: GridRoute NL",
          ["eval.py", "--model", "gemma4-e4b", "--benchmark", "gridroute-nl", "--grid_size", "5",
           "--n", "50", "--output_dir", "./results/eval"], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 1: single-format (GridRoute NL only) on Gemma 4 E2B

SFT warm-start, a short GRPO timing test, then a full run sized to fit `GRPO_BUDGET_MINUTES` at the timing test's measured per-step cost -- see the run-infrastructure cell above for why this is computed rather than manually tuned.

In [ ]:
SFT_SINGLE_DIR = "./results/sft_gemma4-e2b_single"
run_stage("SFT: single-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "nl", "--grid_size", "5",
           "--n_tasks", "400", "--epochs", "1", "--output_dir", SFT_SINGLE_DIR], timeout=TIMEOUT_SFT)


In [ ]:
GRPO_SINGLE_TIMING_DIR = "./results/grpo_gemma4-e2b_single_timing"
run_stage("GRPO timing test: single-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_SINGLE_TIMING_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_SINGLE_DIR = "./results/grpo_gemma4-e2b_single"
single_steps = steps_for_budget(GRPO_SINGLE_TIMING_DIR)
run_stage(f"GRPO: single-format full run ({single_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_SINGLE_DIR,
           "--condition", "single", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(single_steps), "--output_dir", GRPO_SINGLE_DIR],
          required_paths=[SFT_SINGLE_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval single-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "mazebench", "--n", "100", "--output_dir", "./results/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval single-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_SINGLE_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_SINGLE_DIR], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 2: mixed-format (NL + token, naive)

Does training on both formats (interleaved, same underlying grids) do better than single-format alone -- for either benchmark?

In [ ]:
SFT_MIXED_DIR = "./results/sft_gemma4-e2b_mixed"
run_stage("SFT: mixed-format (Gemma 4 E2B)",
          ["train_sft.py", "--model", "gemma4-e2b", "--format", "mixed", "--grid_size", "5",
           "--n_tasks", "400", "--epochs", "1", "--output_dir", SFT_MIXED_DIR], timeout=TIMEOUT_SFT)


In [ ]:
GRPO_MIXED_TIMING_DIR = "./results/grpo_gemma4-e2b_mixed_timing"
run_stage("GRPO timing test: mixed-format",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_MIXED_TIMING_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_MIXED_DIR = "./results/grpo_gemma4-e2b_mixed"
mixed_steps = steps_for_budget(GRPO_MIXED_TIMING_DIR)
run_stage(f"GRPO: mixed-format full run ({mixed_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_MIXED_DIR,
           "--condition", "mixed", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(mixed_steps), "--output_dir", GRPO_MIXED_DIR],
          required_paths=[SFT_MIXED_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval mixed-format checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "mazebench", "--n", "100", "--output_dir", "./results/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval mixed-format checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_MIXED_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_MIXED_DIR], timeout=TIMEOUT_EVAL)


## Phase 2, recipe 3: consistency-reward (one candidate recipe, not this project's headline claim)

Adapts Elhady et al.'s cross-lingual consistency-reward mechanism to cross-format spatial reasoning -- try it, report honestly whether it beats `mixed` or not. See `train_grpo.py`'s `make_reward_fn` docstring for the exact mechanism. This condition generates an extra partner completion per reward call (roughly double the per-step cost of single/mixed) -- `steps_for_budget()` accounts for that automatically since it reads THIS condition's own timing test, not single/mixed's.

In [ ]:
SFT_CONSISTENCY_DIR = SFT_MIXED_DIR  # consistency reuses the same mixed-format SFT warm-start
GRPO_CONSISTENCY_TIMING_DIR = "./results/grpo_gemma4-e2b_consistency_timing"
run_stage("GRPO timing test: consistency-reward",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "20", "--max_steps", "20",
           "--output_dir", GRPO_CONSISTENCY_TIMING_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_QUICK)


In [ ]:
GRPO_CONSISTENCY_DIR = "./results/grpo_gemma4-e2b_consistency"
consistency_steps = steps_for_budget(GRPO_CONSISTENCY_TIMING_DIR)
run_stage(f"GRPO: consistency-reward full run ({consistency_steps} steps)",
          ["train_grpo.py", "--model", "gemma4-e2b", "--adapter_path", SFT_CONSISTENCY_DIR,
           "--condition", "consistency", "--grid_size", "5", "--n_tasks", "50",
           "--max_steps", str(consistency_steps), "--output_dir", GRPO_CONSISTENCY_DIR],
          required_paths=[SFT_CONSISTENCY_DIR], timeout=TIMEOUT_GRPO_FULL)


In [ ]:
run_stage("Eval consistency-reward checkpoint: MazeBench",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "mazebench", "--n", "100", "--output_dir", "./results/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)
run_stage("Eval consistency-reward checkpoint: GridRoute NL",
          ["eval.py", "--model", "gemma4-e2b", "--checkpoint", GRPO_CONSISTENCY_DIR,
           "--benchmark", "gridroute-nl", "--grid_size", "5", "--n", "50", "--output_dir", "./results/eval"],
          required_paths=[GRPO_CONSISTENCY_DIR], timeout=TIMEOUT_EVAL)


## Run summary

At-a-glance status of every stage attempted -- read this first when checking back on an unattended run, before digging into the full cell outputs above.

In [ ]:
import pandas as pd

summary_df = pd.DataFrame(stage_results)
if not summary_df.empty:
    ok = (summary_df["status"] == "ok").sum()
    print(f"{ok}/{len(summary_df)} stages completed successfully.\n")
pd.set_option("display.max_colwidth", None)
summary_df


## Aggregate benchmark results into one comparison table

In [ ]:
import glob

rows = []
for path in sorted(glob.glob("./results/eval/*.json")):
    try:
        with open(path) as f:
            d = json.load(f)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Skipping unreadable results file {path}: {e}")
        continue
    label = path.split("/")[-1]
    row = {"file": label, "model": d.get("model"), "checkpoint": d.get("checkpoint") or "(none)",
           "benchmark": d.get("benchmark"), "n": d.get("n")}
    if "mazebench" in d.get("benchmark", ""):
        row["score"] = d.get("accuracy")
        row["used_official_scoring"] = d.get("used_official_scoring")
    else:
        row["valid_rate"] = d.get("valid_rate")
        row["optimal_rate"] = d.get("optimal_rate")
    rows.append(row)

results_df = pd.DataFrame(rows)
if not results_df.empty:
    results_df.to_csv("./results/comparison_table.csv", index=False)
else:
    print("No eval result files found yet in ./results/eval/ -- nothing to aggregate.")
results_df


## Download results

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/results", "zip", "./results")
print("Saved: /kaggle/working/results.zip -- download it from the Kaggle output panel on the right.")
print("Includes stage_log.json (this run's full stage-by-stage status) and comparison_table.csv.")
